# AML Data Preprocessing
- Using the IBM dataset for AML: https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml
- Dataset is generated by IBM Box Generator, models transactions and illicit activities.
- Original dataset for training will be extensively large, for initial stages of thesis, using smaller dataset of 500,000 transactions.
- In the following we will:
1. explore the data
2. determine nodes and edges
3. determine node and edge attributes
4. create visualization using NetworkX, PyVis, or Graph-tool

* Attributes on ACCOUNT
    * Bank account
    * Account balance
    * BIN number
    * Number of transactions (calculated)
    * Receiving Currency
* Attributes on TRANSACTIONS
    * Payment amount
    * Payment Type
    * Payment Currency (based on “receiving currency” of outgoing bank account)
    * Time


##### *-----IMPORT LIBRARIES-----*

In [ ]:
! pip3 install torch numpy pandas matplotlib torch-geometric

In [ ]:
import torch
import time
import random
import hashlib
import datetime
import itertools
import numpy as np
import pandas as pd
import networkx as nx
from pandas import Timestamp
import matplotlib.pyplot as plt
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_networkx

##### *-----DATA EXPLORATION-----*

In [ ]:
filename = "/Users/owhy/Documents/Datasets/HI-Small_Trans.csv"

In [ ]:
data = pd.read_csv(filename)
data.head()

In [ ]:
print(f"---- shape ----\n - {data.shape}")

In [ ]:
print("---- info ----")
data.info()

In [ ]:
print("---- basic calculations ----")
data.describe()

##### *-----Creating Graph-----*

### NODES and EDGES

Nodes = Bank Accounts -- bank account number
* BIN Number
* Receiving Currency

* Number of transactions (degree matrix --> calculated based on incoming and outcoming flows)

Edges = transactions -- payment amount
* Payment Type
* Payment Currency
* Date and Time

* Account Balance (before transactions)

In [ ]:
data

In [ ]:
# list of unique bank accounts
data_unique = data.drop_duplicates(subset=['Account'])
print(f"Number of bank accounts: {len(data_unique)}")
data_unique

In [ ]:
nodes_df = data_unique[['Account','From Bank','Receiving Currency']]
nodes_df

### One-hot encoding & NODE FEATURES

In [ ]:
# Convert non-numeric columns
positions = nodes_df["Receiving Currency"].str.split(",", expand=True) # creating new columns by splitting receiving currency --> all are added
nodes_df["first_position"] = positions[0] # first currency in each row is extracted --> actual currency used and that we want as TRUE
# One-hot encoding 
node_features = pd.concat([nodes_df, pd.get_dummies(nodes_df["first_position"],dtype='int')], axis=1, join='inner') # effectively adds actual currency to dummy variables/columns
node_features.drop(["Receiving Currency", "first_position"], axis=1, inplace=True) # drop the axiliary columns
node_features.head()

# TODO-DONE! conc unnique random number to the end --> maintain uniqueness
# TODO-DONE! node --> feature --> feature2 --> node2 | problem with uniqueness of node embeddings --> add unique value to identify the node feature vector
# TODO-DONE! create random identity vector for each ACCOUNT + add Account and From Bank as separate properties of the node
# TODO-DONE! ultimately normalize From bank
# TODO-DONE! feature matrix -- > receiving currency n-hot encoding (0 and 1) + encoding of From Bank and Account (word2vec)
# TODO add edge features --> look into EDGE LABELED GRAPHS where all nodes and edges have labels
# TODO create init for loading graph --> initial step


In [ ]:
# Normalize 'From Bank' & 'Account'
def normalize(table, new_min=0, new_max=10):
    if len(table.columns) == 1:
        normalized_df = ((table - table.min()) / (table.max() - table.min())) * (new_max - new_min) + new_min
        return normalized_df
    else:
        normalized_df = pd.DataFrame()
        id = 0
        for column in table.columns:
            col_data = table[column]
            if id == 0:
                normalized_df[f'col_{id}'] = ((col_data - col_data.min()) / (col_data.max() - col_data.min())) * (new_max - new_min) + new_min
            else:
                normalized_column = ((col_data - col_data.min()) / (col_data.max() - col_data.min())) * (new_max - new_min) + new_min
                normalized_df[f'col_{id}'] = normalized_column
            id += 1
        # print(normalized_df)
        return normalized_df

def hashing_vectorization(strings, vector_size=9):
    vectors = []
    for string in strings:
        # Hash the string using hash()
        hashed_values = hash(string) % (10 ** vector_size)  # Ensures unique representations within the specified range
        
        # Convert hashed values to a fixed-size vector
        vector = [int(digit) for digit in str(hashed_values)]
        
        # Ensure vector has the desired size by zero-padding or truncating
        if len(vector) < vector_size:
            vector = [0] * (vector_size - len(vector)) + vector
        elif len(vector) > vector_size:
            vector = vector[:vector_size]
        
        vectors.append(vector)
    
    return vectors

In [ ]:
from_bank_col = node_features.pop('From Bank')
account_col = node_features.pop('Account')

In [ ]:
# node labels for later use

node_labels = pd.DataFrame(account_col)
node_labels

In [ ]:
# TODO add vectors as individuals columns in new dataframe

df = pd.DataFrame(account_col, columns=['Account'])
df.reset_index(drop=True, inplace=True) # Ensure the DataFrame has the same number of rows as the original series
vectors = hashing_vectorization(df['Account'], vector_size=9) 
print(vectors)

# Convert vectors into DataFrame
vectors_df = pd.DataFrame(vectors, columns=[f'col_{i}' for i in range(len(vectors[0]))])
result_df = pd.concat([df, vectors_df], axis=1)

In [ ]:
vectors

In [ ]:
accounts_df = result_df.drop(columns=["Account"])

In [ ]:
from_bank_col

In [ ]:
from_bank_binary = [bin(x).split("b")[1] for x in from_bank_col]
# vectors_df = pd.DataFrame(vectors, columns=[f'col_{i}' for i in range(len(vectors[0]))])

In [ ]:
res = max(from_bank_binary, key=len) 
print("Longest String is  : ", res)
len(res)

In [ ]:
from_bank_binary

In [ ]:
def make_binary_fixed_length(binary_lists, res):
    new_binary_list = []
    for x in binary_lists:
        # print(x)
        if len(x) < len(res):
            num_zeros = len(res) - len(x)
            x = [0] * num_zeros + x
            # print(x)
            new_binary_list.append(x)
        else:
            new_binary_list.append(x)
    return new_binary_list

In [ ]:
binary_lists = [[int(bit) for bit in binary] for binary in from_bank_binary]

In [ ]:
binary_lists = make_binary_fixed_length(binary_lists, res)

In [ ]:
binary_lists

In [ ]:
# Convert vectors into DataFrame
bin_vectors_df = pd.DataFrame(binary_lists, columns=[f'bin_{i}' for i in range(len(binary_lists[0]))])

In [ ]:
bin_vectors_df

In [ ]:
# TODO normalize vector values to avoid big numbers

from_bank_df = pd.DataFrame(from_bank_col)
accounts_df = pd.DataFrame(accounts_df)

# from_bank_df_norm = normalize(from_bank_df,0,1) # TODO do not normalize at this point --> create BINARY representation
accounts_df_norm = normalize(accounts_df,0,1)


In [ ]:
accounts_df_norm

In [ ]:
node_features.reset_index(drop=True, inplace=True) # Ensure the DataFrame has the same number of rows as the original series
accounts_df_norm.reset_index(drop=True, inplace=True) # Ensure the DataFrame has the same number of rows as the original series

node_features = pd.concat([node_features, accounts_df_norm], axis=1)
node_features = pd.concat([node_features, bin_vectors_df], axis=1)
node_features

#### Unique Identifier

In [ ]:
# TODO add unique random identified

unique_ids_set = set()

while len(unique_ids_set) < len(node_features): # uniqueness kept
    unique_ids_set.add(random.random())

unique_ids = list(unique_ids_set)

node_features.insert(0, "Unique ID", unique_ids)

In [ ]:
node_features

### Node Feature Matrix 

In [ ]:
# TODO nodes should be bank accounts and not transactions. Bank accounts have unique receiving currencies and "bank BINs"
x = node_features.to_numpy()
x.shape # [num_nodes x num_features]


### One-hot encoding & EDGE FEATURES

In [ ]:
# TODO add edge features --> look into EDGE LABELED GRAPHS where all nodes and edges have labels
# TODO create init for loading graph --> initial step
# TODO add 2 levels of depth --> will be interconnected, no need to do this step

In [ ]:
links = [(source, destination) for source, destination in zip(data['Account'], data['Account.1'])]

In [ ]:
links

In [ ]:
def get_links(account):
    account_links = []
    for link in links:
        if account in link:
            account_links.append(link)
    return account_links

#### Findings individual links for each accounts

In [ ]:
# VERY INEFFICIENT!!!!!!!!!!!!!!!!!
# VERY INEFFICIENT!!!!!!!!!!!!!!!!!
# VERY INEFFICIENT!!!!!!!!!!!!!!!!!
# VERY INEFFICIENT!!!!!!!!!!!!!!!!!

# TODO different mechanism

nodes = nodes_df['Account']
transaction_limit = 50
nodes = nodes[:transaction_limit] # limit to x instances
all_edges = []

for node in nodes:
    node_df = data[data['Account'] == node] # TODO add 2 levels of depth
    account_links = get_links(node)
    edge_from = [e[0] for e in account_links]
    edge_to = [e[1] for e in account_links]

    print(account_links[:4])
    print("from node --> ", edge_from[:4], "\nto node --> ", edge_to[:4])

    # node_edges = np.column_stack([edge_from, edge_to])
    # all_edges = np.vstack([all_edges, node_edges])

    node_edges = [(edge_from.index(src), edge_to.index(dst)) for src, dst in zip(edge_from, edge_to)]
    all_edges.extend(node_edges)

In [ ]:
# Convert edges to NumPy array
edge_index = np.array(all_edges, dtype=np.int64)
print(edge_index[0:50]) # row 1: Source node 0 (the central node) has an outgoing edge to itself (destination node 0)

### Edge Features

In [ ]:
# TODO add edge features --> create matrix like those for nodes

edges_df = data[["Timestamp", "Amount Paid", "Payment Currency", "Payment Format"]]
edges_df

### Payment Encoding

In [ ]:
edges_amount = edges_df["Amount Paid"].astype(str)
edges_amount = list(edges_amount)

In [ ]:
def count_unused_decimals(number):
    # Convert number to string to iterate through digits
    num_str = str(number)
    count = 0

    # Iterate through digits from the end
    for digit in reversed(num_str):
        # If the digit is '0', increment count
        if digit == '0':
            count += 1
        # If non-zero digit encountered, break the loop
        else:
            break
    
    # Remove trailing zeroes from the number
    num_str = num_str.rstrip('0')

    return num_str, count

In [ ]:
maximum = str(max(edges_df["Amount Paid"]))
max_len = len(maximum.split(".")[0])

minimum = min(edges_df["Amount Paid"])
minimum = format(minimum, 'f')
min_len = len(str(minimum.split('.')[1]))

new_min, count = count_unused_decimals(minimum)
min_len = min_len - count

number_columns = max_len + min_len
number_columns

In [ ]:
def split_into_vectors(table):
    lists = []
    for binary in table:
        binary = float(binary)
        binary = str(format(binary, 'f'))
        decimal_repr = []
        for bit in binary:
            if '.' not in bit:
                decimal_repr.append(str(int(bit)))
            else:
                decimal_repr.append(bit)
        lists.append(decimal_repr)
    return lists

In [ ]:
def encode_payment_amount(df_col, max_len, min_len):
    new_payment_list = []
    for x in df_col:
        # print(x)
        index_of_decimal = x.index('.')
        positive_decimals = index_of_decimal
        negative_decimals = len(x) - (index_of_decimal+1)
        if positive_decimals < max_len:
            num_zeros = max_len - positive_decimals
            x = ['0'] * num_zeros + x
            # print(x)
            new_payment_list.append(x)
        elif negative_decimals < min_len:
            num_zeros = max_len - negative_decimals
            x = ['0'] * num_zeros + x
            # print(x)
            new_payment_list.append(x)
        else:
            new_payment_list.append(x)
        x.remove('.')
    return new_payment_list

In [ ]:
a = split_into_vectors(edges_amount)

In [ ]:
new_payment_list = encode_payment_amount(a, max_len, min_len)

In [ ]:
new_payment_list = nested_list_int = [[int(item) for item in sublist] for sublist in new_payment_list]

In [ ]:
new_payment_list

In [ ]:
# Convert vectors into DataFrame
payment_vectors_df = pd.DataFrame(new_payment_list, columns=[f'payment_{i}' for i in range(len(new_payment_list[0]))])

In [ ]:
edges_featues = pd.concat([edges_df, payment_vectors_df], axis=1)
edges_featues.drop("Amount Paid", axis='columns')

In [ ]:
# TODO convert Currency into one-hot encoding

positions = edges_featues["Payment Currency"].str.split(",", expand=True) # creating new columns by splitting receiving currency --> all are added
edges_featues["first_position"] = positions[0] # first currency in each row is extracted --> actual currency used and that we want as TRUE
# One-hot encoding 
edges_features = pd.concat([edges_featues, pd.get_dummies(edges_featues["first_position"],dtype='int')], axis=1, join='inner') # effectively adds actual currency to dummy variables/columns
edges_features.drop(["Amount Paid","Payment Currency", "first_position"], axis=1, inplace=True) # drop the axiliary columns
edges_features.head()

In [ ]:
# TODO convert Payment Format

positions_2 = edges_features["Payment Format"].str.split(",", expand=True) 
edges_features["second_position"] = positions_2[0]
edges_features = pd.concat([edges_features, pd.get_dummies(edges_features["second_position"],dtype='int')], axis=1, join='inner') # effectively adds actual currency to dummy variables/columns
edges_features.drop(["Payment Format", "second_position"], axis=1, inplace=True) # drop the axiliary columns
edges_features.head()

In [ ]:
# TODO convert timestamps

edges_features["Timestamp"] = pd.to_datetime(edges_features['Timestamp']).astype(int) // 10**9 # does not interpret time well... circular definition for months --> sinus calculations

In [ ]:
edges_features.head()

### Graphical Representation

In [ ]:
edge_attr = edges_features.to_numpy()
print(edge_attr[0:10])

In [ ]:
# TODO label edge

import networkx as nx

graph1 = nx.Graph()

for i in range(len(edge_index)):
    u = edge_index[i][0]
    v = edge_index[i][1]
    graph1.add_edge(u,v,label=edge_attr[i][1]) # use edge labels for edge features?

print(graph1.edges(data=True))

In [ ]:
pos = nx.spring_layout(graph1) # shell, circular, spectral, spring, random, 

plt.figure(figsize=(25, 15))  # Increase figure size

nx.draw(
    graph1, 
    pos, 
    node_size=650,  # Reduce node size for better visibility
    with_labels=True, 
    font_size=15, 
    font_weight='bold', 
    node_color='lightblue',  # Specify node color
    edge_color='gray',  # Specify edge color
    width=1,  # Adjust edge width
    arrows=True,  # Show arrows for directed edges
    arrowstyle='->',  # Specify arrow style
    arrowsize=30,  # Adjust arrow size
)

edge_labels = nx.get_edge_attributes(graph1, 'label')
nx.draw_networkx_edge_labels(
    graph1, 
    pos, 
    edge_labels=edge_labels, 
    label_pos=0.5,  # Adjust label position along edges
    font_size=7,  # Adjust font size
    font_color='green',  # Specify font color
)

plt.title(f'Graph Visualization of first {transaction_limit} transactions')  # Add title to the plot
plt.axis('off')  # Hide axis
plt.show()

In [ ]:
print(nx.degree_centrality(graph1)) # closeness_centrality, eigenvector_centrality, betweeness_centrality
print(nx.betweenness_centrality(graph1)) 

In [ ]:
# TODO add edge and node features to graph --- https://www.youtube.com/watch?v=TlkpoB3JAHE&ab_channel=Koolac



## *** Questions: ****

##### - How specifically do I need to add features to the nodes and edges?
##### - Nodes are not added directly, assume I need to add them directly as well.
##### - Very inefficient node link mechanism --> took 10 seconds for 50 transactions --> 5 mil transactions will take me 277 hours... need to change method
##### - Is there a need to implement anything specific for a directed graph?
##### - going to the next step --> creating value for each relation, to be used in function
##### - Link prediction using node2vec???

In [ ]:
"""
NEXT 2 WEEKS:

1. Vectorize "BANKS" - BINs using binary encoding --> DONE!!!
2. Vectorize "Amount Paid" --> DONE!!!

3. Mapping through dictionary of all links --> No need to find individual links between accounts --> we are taking all accounts and the graph in general
4. Graph Visualization should include labels that are actual accounts
5. Add statistics to feature matrix X? --> if necessary
6. Apply graph on subset of 1000 transactions
7. Create Adjacency Matrix --> can be done through networkx or through existing links
8. Create GNN model
9. Write introduction
10. Write literature review

"""

In [ ]:
# What is node2vec?

# TODO BINs sould also be vectorized to avoid ordering
# TODO binary encoding for banks --> replace 300 features with 10 features that can represent a number in binary 
# TODO take random subset from transactions

# TODO calculating time --> circular definition for months (goes back to 0) --> sinus calculations --> same time different year cannot be distinguished in unix format
# TODO --> don't do yet.

# TODO payment amount --> separate based on power of 10s --> separate columns for thousands, hundreds etc.

# TODO no need to add nodes in order --> mapping thorugh dictionary is an option

# TODO potentially add statistics to feature matrix X


## SIMPLE GNN MODEL EXAMPLE

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
num_features =
hidden_dim =
num_classes =
num_epochs =
features =
adj =
labels =

In [ ]:
class GCNBlock(nn.Module):
    def __init__(self, in_features, out_features):
        super(GCNBlock, self).__init__()
        self.linear = nn.Linear(in_features, out_features)
        
    def forward(self, x, adj):
        x = self.linear(x)
        x = torch.matmul(adj, x)
        x = F.relu(x)
        return x
class GCN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(GCN, self).__init__()
        self.gcn1 = GCNBlock(input_dim, hidden_dim)
        self.gcn2 = GCNBlock(hidden_dim, output_dim)
        
    def forward(self, x, adj):
        x = self.gcn1(x, adj)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.gcn2(x, adj)
        return x
# Define the model
model = GCN(num_features, hidden_dim, num_classes)
# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
# Train the model
for epoch in range(num_epochs):
    optimizer.zero_grad()
    outputs = model(features, adj)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

In [ ]:
import datetime
import os
from typing import Callable, Optional
import pandas as pd
from sklearn import preprocessing
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

import torch

from torch_geometric.data import Data, InMemoryDataset

pd.set_option("display.max_columns", None)


class AMLtoGraph(InMemoryDataset):

    def __init__(
        self,
        root: str,
        edge_window_size: int = 10,
        transform: Optional[Callable] = None,
        pre_transform: Optional[Callable] = None,
    ):
        self.edge_window_size = edge_window_size
        super().__init__(root, transform, pre_transform)
        self.data, self.slices = torch.load(self.processed_paths[0])

    @property
    def raw_file_names(self) -> str:
        return "HI-Small_Trans.csv"

    @property
    def processed_file_names(self) -> str:
        return "data.pt"

    @property
    def num_nodes(self) -> int:
        return self._data.edge_index.max().item() + 1

    def df_label_encoder(self, df, columns):
        le = preprocessing.LabelEncoder()
        for i in columns:
            df[i] = le.fit_transform(df[i].astype(str))
        return df

    def preprocess(self, df):
        df = self.df_label_encoder(
            df, ["Payment Format", "Payment Currency", "Receiving Currency"]
        )
        df["Timestamp"] = pd.to_datetime(df["Timestamp"])
        df["Timestamp"] = df["Timestamp"].apply(lambda x: x.value)
        df["Timestamp"] = (df["Timestamp"] - df["Timestamp"].min()) / (
            df["Timestamp"].max() - df["Timestamp"].min()
        )

        df["Account"] = df["From Bank"].astype(str) + "_" + df["Account"]
        df["Account.1"] = df["To Bank"].astype(str) + "_" + df["Account.1"]
        df = df.sort_values(by=["Account"])
        receiving_df = df[["Account.1", "Amount Received", "Receiving Currency"]]
        paying_df = df[["Account", "Amount Paid", "Payment Currency"]]
        receiving_df = receiving_df.rename({"Account.1": "Account"}, axis=1)
        currency_ls = sorted(df["Receiving Currency"].unique())

        return df, receiving_df, paying_df, currency_ls

    def get_all_account(self, df):
        ldf = df[["Account", "From Bank"]]
        rdf = df[["Account.1", "To Bank"]]
        suspicious = df[df["Is Laundering"] == 1]
        s1 = suspicious[["Account", "Is Laundering"]]
        s2 = suspicious[["Account.1", "Is Laundering"]]
        s2 = s2.rename({"Account.1": "Account"}, axis=1)
        suspicious = pd.concat([s1, s2], join="outer")
        suspicious = suspicious.drop_duplicates()

        ldf = ldf.rename({"From Bank": "Bank"}, axis=1)
        rdf = rdf.rename({"Account.1": "Account", "To Bank": "Bank"}, axis=1)
        df = pd.concat([ldf, rdf], join="outer")
        df = df.drop_duplicates()

        df["Is Laundering"] = 0
        df.set_index("Account", inplace=True)
        df.update(suspicious.set_index("Account"))
        df = df.reset_index()
        return df

    def paid_currency_aggregate(self, currency_ls, paying_df, accounts):
        for i in currency_ls:
            temp = paying_df[paying_df["Payment Currency"] == i]
            accounts["avg paid " + str(i)] = (
                temp["Amount Paid"].groupby(temp["Account"]).transform("mean")
            )
        return accounts

    def received_currency_aggregate(self, currency_ls, receiving_df, accounts):
        for i in currency_ls:
            temp = receiving_df[receiving_df["Receiving Currency"] == i]
            accounts["avg received " + str(i)] = (
                temp["Amount Received"].groupby(temp["Account"]).transform("mean")
            )
        accounts = accounts.fillna(0)
        return accounts

    def get_edge_df(self, accounts, df):
        accounts = accounts.reset_index(drop=True)
        accounts["ID"] = accounts.index
        mapping_dict = dict(zip(accounts["Account"], accounts["ID"]))
        df["From"] = df["Account"].map(mapping_dict)
        df["To"] = df["Account.1"].map(mapping_dict)
        df = df.drop(["Account", "Account.1", "From Bank", "To Bank", "Payment Format", "Payment Currency"], axis=1)

        edge_index = torch.stack(
            [torch.from_numpy(df["From"].values), torch.from_numpy(df["To"].values)],
            dim=0,
        )

        df = df.drop(["Is Laundering", "From", "To"], axis=1)

        edge_attr = torch.from_numpy(df.values).to(torch.float)
        return edge_attr, edge_index

    def get_node_attr(self, currency_ls, paying_df, receiving_df, accounts):
        node_df = self.paid_currency_aggregate(currency_ls, paying_df, accounts)
        node_df = self.received_currency_aggregate(currency_ls, receiving_df, node_df)
        node_label = torch.from_numpy(node_df["Is Laundering"].values).to(torch.float)
        node_df = node_df.drop(["Account", "Is Laundering"], axis=1)
        node_df = self.df_label_encoder(node_df, ["Bank"])
        node_df = torch.from_numpy(node_df.values).to(torch.float)
        return node_df, node_label

In [ ]:
dataset = AMLtoGraph("/Users/owhy/Downloads")
print(type(dataset))

In [ ]:
dataset[0]

In [ ]:
print("Node features shape:", example.x.shape)
print("Edge index shape:", example.edge_index.shape)
print("Edge attributes shape:", example.edge_attr.shape)
print("Labels shape:", example.y.shape)

In [ ]:
# Convert the PyTorch Geometric Data object to a NetworkX graph
def to_networkx(data, node_id, num_transactions=10):
    G = nx.Graph()

    # Add the central node
    G.add_node(node_id, label=f"Account: {node_id}", color="blue", size=500)

    # Get the indices of transactions related to the given node
    related_edges = (data.edge_index[0] == node_id).nonzero().squeeze()
    num_related_edges = min(num_transactions, related_edges.numel())

    # Add the related transactions and corresponding nodes
    for i in range(num_related_edges):
        transaction_id = related_edges[i].item()
        source_node_id = data.edge_index[0, transaction_id].item()
        target_node_id = data.edge_index[1, transaction_id].item()
        transaction_amount = data.edge_attr[transaction_id, 0].item()

        # Add the transaction edge
        G.add_edge(
            source_node_id,
            target_node_id,
            label=f"Amount: {transaction_amount}",
            color="gray",
        )

        # Add the source and target nodes
        if source_node_id != node_id:
            G.add_node(
                source_node_id,
                label=f"Account: {source_node_id}",
                color="green",
                size=300,
            )
        if target_node_id != node_id:
            G.add_node(
                target_node_id,
                label=f"Account: {target_node_id}",
                color="red",
                size=300,
            )

    return G


# Visualize the NetworkX graph
def visualize_networkx(graph):
    pos = nx.spring_layout(graph, seed=42)  # Compute layout

    # Draw nodes
    node_colors = [graph.nodes[n]["color"] for n in graph.nodes()]
    node_sizes = [graph.nodes[n]["size"] for n in graph.nodes()]
    nx.draw_networkx_nodes(graph, pos, node_color=node_colors, node_size=node_sizes)

    # Draw edges
    edge_colors = [graph.edges[e]["color"] for e in graph.edges()]
    nx.draw_networkx_edges(graph, pos, width=1.0, alpha=0.5, edge_color=edge_colors)

    # Draw labels
    node_labels = nx.get_node_attributes(graph, "label")
    nx.draw_networkx_labels(
        graph, pos, labels=node_labels, font_size=8, font_color="black"
    )
    edge_labels = nx.get_edge_attributes(graph, "label")
    nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels, font_size=8)

    # Display the graph
    plt.title("Graph Visualization")
    plt.axis("off")
    plt.show()


# Convert the first example in the dataset to a NetworkX graph focused on a specific node
example_index = 0
example_data = dataset[example_index]
node_id_to_visualize = 1  # ID of the node to visualize
num_transactions_to_visualize = 10  # Number of transactions to visualize
graph = to_networkx(example_data, node_id_to_visualize, num_transactions_to_visualize)

# Visualize the NetworkX graph
visualize_networkx(graph)